# ⚡ Velmire Clip AI - Free GPU Colab Backend Server
Jalankan notebook ini di **Google Colab (Gratis GPU)** untuk memproses klip YouTube dengan cepat dan menghasilkan URL Ngrok untuk aplikasi mobile **Velmire Clip**.

In [ ]:
# 1. Install Dependencies & FFmpeg
!apt-get update -qq && !apt-get install -y ffmpeg
!pip install flask flask-cors pyngrok yt-dlp

In [ ]:
# 2. Server Python Colab Engine
import os, uuid, subprocess
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

OS_DIR = './output'
TEMP_DIR = './temp'
os.makedirs(OS_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

jobs = {}

@app.route('/api/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'server': 'Colab GPU'})

@app.route('/api/clip', methods=['POST'])
def create_clip():
    data = request.json
    url = data.get('url')
    start_sec = data.get('startTimeSec', 0)
    end_sec = data.get('endTimeSec', 30)
    ratio = data.get('ratio', '9:16')
    sub_style = data.get('subtitleStyle', 'auto')

    job_id = str(uuid.uuid4())
    output_filename = f'clip_{job_id[:8]}.mp4'
    output_path = os.path.join(OS_DIR, output_filename)

    jobs[job_id] = {
        'id': job_id,
        'status': 'downloading',
        'statusMessage': 'downloading...',
        'outputUrl': None,
        'error': None
    }

    # Run processing
    def process():
        raw_path = os.path.join(TEMP_DIR, f'raw_{job_id}.mp4')
        try:
            # Download
            cmd_dl = f'yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best" --no-playlist -o "{raw_path}" "{url}"'
            subprocess.run(cmd_dl, shell=True, check=True)

            # Crop
            jobs[job_id]['statusMessage'] = 'clipping & cropping...'
            duration = max(1, end_sec - start_sec)
            crop_filter = 'crop=ih*9/16:ih:(iw-ih*9/16)/2:0,scale=720:1280' if ratio == '9:16' else 'scale=1280:720'

            cmd_ff = f'ffmpeg -ss {start_sec} -i "{raw_path}" -t {duration} -vf "{crop_filter}" -c:v libx264 -preset ultrafast -c:a aac "{output_path}" -y'
            subprocess.run(cmd_ff, shell=True, check=True)

            jobs[job_id]['status'] = 'selesai'
            jobs[job_id]['statusMessage'] = 'selesai'
            jobs[job_id]['outputUrl'] = f'/output/{output_filename}'
        except Exception as e:
            jobs[job_id]['status'] = 'failed'
            jobs[job_id]['error'] = str(e)
        finally:
            if os.path.exists(raw_path): os.remove(raw_path)

    import threading
    threading.Thread(target=process).start()
    return jsonify({'jobId': job_id})

@app.route('/api/status/<job_id>', methods=['GET'])
def status(job_id):
    return jsonify(jobs.get(job_id, {'error': 'not found'}))

@app.route('/output/<filename>', methods=['GET'])
def get_output(filename):
    return send_from_directory(OS_DIR, filename)

# Start Ngrok Tunnel
public_url = ngrok.connect(5000).public_url
print('====================================================')
print('🚀 COLAB BACKEND BERHASIL DIJALANKAN!')
print('Copy URL ini dan masukkan ke aplikasi Velmire:')
print(public_url)
print('====================================================')

app.run(port=5000)